# Cosmic web (Quijote 2D): data and MGD model

Model `base_real_np400`: L_6 + scattering orders 1, 2, 3 (Real), r = 324 statistics, 128 x 128 maps (log density, db4
coarse-grained, standardised), two_phase grid nt = 3000, **N = 400 walkers** (each of the 100 data maps 4 times; N - 1 >= r
so the time term of Theta_reg is full rank, see the 2026-11-09 analysis). Theta = Theta_reg(t = 1) at lam = 1e-3
(cross-seed coherence of log p 0.999 on the plateau 1e-3 .. 1e-2).

Conventions (as the other talk notebooks): -log p in every log-p figure, natural log for nats, u = -log10 p / d with
d = 128^2 for histograms and tables; bands: data = min-max over 5 batches of 20 maps, model = mean +- std over seeds
(not like-for-like). Maps are standardised: heights and thresholds in data std.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()                                   # cosmic_web/
for p in (ROOT, ROOT.parent, ROOT.parent / 'codes'):
    sys.path.insert(0, str(p))
import numpy as np
import pandas as pd
import torch
from IPython.display import display
import cw_common as cw
import cw_figures as cwf
import talk_figures as tf
tf.set_output(ROOT / 'figures' / 'talk_cosmic')      # every figure is saved here as PNG and shown as an image (any backend)

ARM = 'base_real_np400'
SEEDS = [900, 901, 902]
LAM = 1e-3
THETA_DIR = None                                     # default: $SCRATCH/MGD-for-Maximum-Entropy-Generation/cosmic_web/theta_reg/<ARM>
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
R = cwf.load_run(ARM, SEEDS, LAM, THETA_DIR, device=DEVICE)

## Data and generated maps

In [ ]:
cwf.plot_maps(R, n=4)

## Statistics not imposed by the model

### Pixel PDF and isotropic power spectrum
The power spectrum is only partly constrained (through the second-order scattering moments); the PDF is not imposed.

In [ ]:
cwf.plot_pdf_spectrum(R)

### Wavelet coefficients per scale
Histograms of Re W_j (Morlet bank of the model, J = 5, L = 4, orientations pooled), each scale divided by the data std
of Re W_j. The printed table is the centre |k| of each band-pass filter (check that the index order is scale-major).

In [ ]:
cwf.plot_wavelet_hist(R)

### Peak counts
Local maxima (3 x 3 neighbourhood, periodic), per map and per unit height.

In [ ]:
cwf.plot_peaks(R)

### Minkowski functionals
Excursion sets {x > nu}: area fraction, boundary length (pixel-edge estimator, overestimates an isotropic length by
4/pi for both data and model), Euler characteristic (vertices - edges + faces of the union of closed pixels).

In [ ]:
cwf.plot_minkowski(R)

## Imposed moments (in-sample check)
Offset |data mean - generated mean| / data std of each of the r imposed statistics, against the offset independent
sampling would give. MGD pins the walkers' empirical mean to the data mean, so this only confirms the SDE worked.

In [ ]:
T_mom = cwf.moment_check(R, device=DEVICE)
display(T_mom)

## Entropy bound
Final MGD entropy bound (a lower bound on the entropy of the model) and entropy of the Gaussian with the data
covariance; gap per pixel in nats.

In [ ]:
display(cwf.entropy_table(R))

## Number of statistics in the model

In [ ]:
pots = cw.build_potentials(R['cfg'], 'cpu')
display(tf.count_statistics(pots, R['x_data'], R['theta'][SEEDS[0]]))

## Distribution of -log p
u = -log10 p / d, each seed centred on its own median (log p is known up to a constant per seed), seeds pooled.
Generated maps (samples of the model) and the 100 data maps (scored by each seed's Theta).

In [ ]:
def pooled(C):
    return np.concatenate([c - np.median(c) for c in C['TOTAL']])

tf.plot_logp_histogram(pooled(R['C_gen']), 'neglogp_generated', d=R['d'], center='median', bins=60)
tf.plot_logp_histogram(pooled(R['C_data']), 'neglogp_data', d=R['d'], center='median', bins=30)

## -log p of the generated maps vs a Gaussian field
For a Gaussian field's own samples, -ln p = chi2_d / 2 + const whatever the covariance (spectrum), so the reference is
the analytic chi2_d / 2 density (std sqrt(d/2) = 90.5 nats at d = 128^2); no spectrum is needed.

In [ ]:
sd_mgd, sd_gauss = cwf.plot_logp_vs_chi2(R)
print('std of -log p (nats): MGD per seed', sd_mgd.round(1), '| Gaussian', round(sd_gauss, 1))

## Contribution of each potential family to -log p (generated maps)

In [ ]:
tf.plot_energy_bars(R['C_gen'], 'energy_bars', title='Cosmic web')

## Table: Var/2 of -log10 p / d per category (generated maps)
Var(ln p_k) / (2 d ln 10), digits per pixel (first-order KL of each category); confinement (L_6) in its own table.

In [ ]:
T_kl = tf.kl_table(R['C_gen'], R['widths'], R['d'])
tf.plot_kl_table_single(T_kl, 'kl_table', 'Cosmic web')
tf.plot_confinement_table({'Cosmic web': T_kl}, 'confinement_table')
display(T_kl)

## Exact D_k (professor's mail)
D_k = D_KL(p || q_k) = ln E_p exp(-(C_k - E_p C_k)), q_k = p without the category's term, renormalised; empirical mean
over the generated maps (samples of p). Next to it the first-order term Var(C_k)/2. ESS = effective number of maps in
the exponential average: a small ESS means D_k is set by a few maps (then the estimate is unreliable).

In [ ]:
display(cwf.dk_table(R, source='gen'))
display(cwf.dk_table(R, source='data'))